# ДЗ-1. Ваш участок ПРАЙМ

Вы — аналитик команды подписки ПРАЙМ. У каждого аналитика свой **участок**: одна таблица, один сегмент и одно окно времени. Это тот самый персональный срез из `prime.assignments`, с которым вы работали на первом семинаре, и больше он ни у кого не повторяется.

В понедельник руководитель просит сводку по участку — восемь задач. В каждой две части: **расчёт** (функция на Python) и **вопрос**, на который вы отвечаете словами в ячейке «✍️ Ответ на вопрос N». Числа у всех разные, поэтому и выводы у каждого свои.

### Как устроен ноутбук

| Часть | Что там | Что делаете вы |
|---|---|---|
| **1. Подготовка** | готовый код: подключение к базе и выгрузка вашего участка | вписываете пять значений в ячейку 1.1 и запускаете ячейки по порядку |
| **2. Восемь задач** | условия, заготовки функций, вопросы | пишете код функций и ответы словами |
| **3. Итог** | готовый код: сводка ваших ответов и проверка их формата | запускаете перед сдачей |
| **4. Исследование** | две открытые задачи для тех, кто хочет 9 или 10 | по желанию |

**Оценка.** Части 1–3 — это оценка до 8 баллов: сделали всё аккуратно, выводы верные — 8, и это «отлично». 9 и 10 — только за исследование в части 4. Все критерии — в [тексте задания](https://github.com/tikhomirovd/python-for-ba-hse-2026/tree/master/задания/дз-1-окружение-и-репозиторий), там же — как сдавать.

Запускайте ячейки сверху вниз (`Shift + Enter`). Перед сдачей — **Kernel → Restart Kernel and Run All Cells**: ноутбук должен пройти целиком без красного.

Ноутбук лежит в **вашем** репозитории как `notebooks/hw1.ipynb`, а JupyterLab запускается из корня `prime-monitor` командой `uv run jupyter lab`.

## Часть 1. Подготовка — готовый код

Код в этой части писать не нужно — он готов. Единственная ваша правка — пять значений в ячейке 1.1. Номер ячейки — это номер заголовка над ней: «ячейка 1.1» — код сразу под заголовком «1.1 Ваш участок».

Код длинный, потому что аккуратно обходит несколько ловушек, до которых мы дойдём позже. Прочитать его полезно, разбирать каждую строку не обязательно.

| Ячейка | Что делает | Что делаете вы |
|---|---|---|
| 1.1 Ваш участок | хранит пять значений участка | вписываете значения |
| 1.2 Подключение | подключается к базе через ваш `.env` | запускаете |
| 1.3 Строка участка | показывает вашу строку из `prime.assignments` | запускаете, копируете значения в 1.1 и запускаете 1.1 ещё раз |
| 1.4 Запрос | описывает, как достать участок из любой из четырёх таблиц | запускаете |
| 1.5 Выгрузка | достаёт участок в переменную `rows` | запускаете — дальше вы работаете с `rows` |
| 1.6 Формат ответов | служебная функция для итога | запускаете |

### 1.1 Ваш участок

Впишите значения из своей строки `prime.assignments` — строками, в кавычках. Какие они, покажет ячейка 1.3: запустите 1.2 и 1.3, скопируйте напечатанные строки сюда и **запустите эту ячейку ещё раз**, иначе Python продолжит видеть старые значения.

Это **единственное место**, где указан участок. При проверке преподаватель подставит сюда другой участок — и все ответы должны пересчитаться сами. Поэтому ячейку не удаляйте и не пересоздавайте, а значения больше нигде не повторяйте.

In [ ]:
TABLE = "transactions"         # таблица, например "payments"
SEGMENT_COLUMN = "channel"     # колонка сегмента, например "payment_method"
SEGMENT = "web"                # сегмент — значение в этой колонке, например "card"
PERIOD_START = '2025-04-01'    # первый день окна, например "2026-04-01"
PERIOD_END = '2025-04-30'      # последний день окна ВКЛЮЧИТЕЛЬНО, например "2026-06-30"

### 1.2 Подключение

Подключаемся к учебной базе и заводим функцию `query(sql)`: она отправляет SQL-запрос и возвращает результат списком словарей — одна строка результата, один словарь, как на семинарах. Пароля в ноутбуке нет: его знает только ваш `.env`.

Если здесь ошибка `Пакет prime не найден` — ноутбук запущен не в окружении вашего репозитория. Если `Не найдена переменная PRIME_DSN` — нет файла `.env` в корне `prime-monitor`.

Соединение открывается на каждый запрос и сразу закрывается: так после сна ноутбука или смены Wi-Fi ничего не отваливается, а общий сервер не держит лишних подключений.

In [ ]:
import sys
from collections import Counter, defaultdict  # понадобятся в задачах
from datetime import date
from decimal import Decimal

from sqlalchemy import text

# get_engine() живёт в вашем репозитории, в src/prime/config.py.
# Она читает строку подключения из .env — поэтому пароля здесь нет.
try:
    from prime.config import get_engine
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "Пакет prime не найден: ноутбук запущен не в окружении prime-monitor.\n"
        f"Сейчас Python отсюда: {sys.executable}\n"
        "JupyterLab: закройте его, перейдите в корень prime-monitor и выполните uv run jupyter lab.\n"
        "VS Code: Select Kernel справа сверху -> .venv из папки prime-monitor."
    ) from None

engine = get_engine()


def query(sql: str, **params) -> list[dict]:
    """Выполнить SQL-запрос и вернуть строки списком словарей {колонка: значение}."""
    try:
        with engine.connect() as conn:
            # Моменты времени в базе хранятся с часовым поясом. День события считаем
            # по UTC — так он у всех одинаковый, какие бы настройки ни стояли у вас.
            conn.execute(text("set time zone 'UTC'"))
            result = conn.execute(text(sql), params)
            return [dict(row) for row in result.mappings()]
    finally:
        # Соединение не держим: сервер у всей группы общий, а после сна ноутбука
        # старое соединение всё равно мёртвое.
        engine.dispose()


print("подключились к базе как:", query("select current_user as login")[0]["login"])

### 1.3 Строка участка

Показывает вашу строку из `prime.assignments` — ту же, что на первом семинаре. Чужих строк база не показывает. Скопируйте напечатанное в ячейку 1.1 и запустите 1.1 ещё раз.

За то, что в 1.1 стоят именно ваши значения, отвечаете вы: код с чужим участком спокойно всё посчитает, только не то.

In [ ]:
mine = query("select * from prime.assignments where login = current_user")

if mine:
    row = mine[0]
    print("Ваш участок. Скопируйте в ячейку 1.1 и запустите её:\n")
    print(f"TABLE = {row['table_name']!r}")
    print(f"SEGMENT_COLUMN = {row['segment_column']!r}")
    print(f"SEGMENT = {row['segment']!r}")
    print(f"PERIOD_START = {str(row['period_start'])!r}")
    print(f"PERIOD_END = {str(row['period_end'])!r}")
else:
    # Так бывает, когда ноутбук запускает преподаватель под своей ролью.
    print("Строки участка для этого пользователя в базе нет.")

### 1.4 Запрос

Четыре таблицы устроены по-разному, но для задач нужно одно и то же. Поэтому запрос приводит любую из них к **пяти полям**:

| Поле | Что это |
|---|---|
| `row_id` | идентификатор строки |
| `actor_id` | участник — тот, чьи это строки |
| `day` | день события, тип `datetime.date` |
| `ok` | состоялось ли событие: `True` или `False` |
| `value` | сколько: деньги, минуты или часы, тип `Decimal` |

Что это значит в **вашей** таблице:

| Таблица | Одна строка — это | Участник | Сегмент | `ok = True` | `ok = False` | `value` |
|---|---|---|---|---|---|---|
| `payments` | попытка списать плату за подписку | подписка | способ оплаты: `card`, `sbp`, `wallet`, `balance` | списание прошло | не прошло или деньги вернули | сумма списания, ₽ |
| `transactions` | покупка клиента в партнёрском сервисе | клиент | канал: `app` — приложение, `web` — сайт, `pos` — касса | покупка прошла | отклонена или проведена и отменена | сумма покупки, ₽; бывает ноль и меньше нуля |
| `service_usage` | сессия в сервисе ПРАЙМ | клиент | устройство | сессия записана корректно: конец позже начала | конец не позже начала | длительность, минуты |
| `support_tickets` | обращение в поддержку | клиент | тема обращения | обращение решено | ещё в работе | время от создания до решения, часы; у нерешённых — `None` |

**Участник** здесь — просто тот, чьи это строки. Не путайте с участниками подписки из `subscription_members`, о которых шла речь на первом семинаре. А у `support_tickets` сумма `value` по дню растёт и от числа обращений, и от медленных решений — это не одно и то же.

Как работает запрос:

1. Блок `acts` выбирает **1000 участников** вашего сегмента за ваше окно. Это выборка, а не весь сегмент: поэтому строк в `rows` меньше, чем вы насчитали в хвосте первого семинара, — там был весь срез. Подгонять под то число не нужно. В ответах словами пишите «в выгрузке» или «среди 1000 подписок / клиентов», а не «по всем картам за квартал». Перемешивание через `md5` делает выбор похожим на случайный, но при каждом запуске — одинаковым.
2. Основной запрос берёт строки этих участников — **только в вашем сегменте и в вашем окне** (тот же фильтр, что в `acts`) — и переименовывает колонки в пять полей.
3. В `{фигурных скобках}` — названия колонок из словаря `COLUMNS`: они подставляются под вашу таблицу. После двоеточия (`:segment`, `:period_start`, `:period_end`) — ваши значения из ячейки 1.1.
4. Граница окна: момент события `>=` первого дня и `<` дня, следующего за последним. Так последний день попадает в окно целиком.

In [ ]:
# Как в каждой из четырёх таблиц называются колонки, из которых получаются пять полей.
COLUMNS = {
    "payments": {
        "segment_column": "payment_method",   # колонка сегмента
        "row_id": "payment_id",                # -> row_id
        "actor": "subscription_id",            # -> actor_id
        "moment": "paid_at",                   # момент события -> day
        "ok": "status = 'success'",            # -> ok
        "value": "amount",                     # -> value
    },
    "transactions": {
        "segment_column": "channel",
        "row_id": "transaction_id",
        "actor": "client_id",
        "moment": "occurred_at",
        "ok": "status = 'success'",
        "value": "amount",
    },
    "service_usage": {
        "segment_column": "device_type",
        "row_id": "usage_id",
        "actor": "client_id",
        "moment": "started_at",
        "ok": "ended_at > started_at",
        "value": "extract(epoch from (ended_at - started_at)) / 60.0",      # секунды -> минуты
    },
    "support_tickets": {
        "segment_column": "category",
        "row_id": "ticket_id",
        "actor": "client_id",
        "moment": "created_at",
        "ok": "resolved_at is not null",
        "value": "extract(epoch from (resolved_at - created_at)) / 3600.0",  # секунды -> часы
    },
}

SLICE_SQL = """
with acts as (
    select actor_id
    from (
        select distinct {actor}::text as actor_id
        from prime.{table}
        where {segment_column} = :segment
          and {moment} >= cast(:period_start as date)
          and {moment} <  cast(:period_end as date) + 1
    ) s
    order by md5(actor_id)
    limit 1000
)
select t.{row_id}::text as row_id,
       t.{actor}::text  as actor_id,
       t.{moment}::date as day,
       {ok}             as ok,
       {value}          as value
from prime.{table} t
join acts a on a.actor_id = t.{actor}::text
where t.{segment_column} = :segment
  and t.{moment} >= cast(:period_start as date)
  and t.{moment} <  cast(:period_end as date) + 1
order by t.{row_id}
"""

print("запрос описан")

### 1.5 Выгрузка

Достаём участок из базы в переменную `rows` — список словарей, по одному на строку. **Все задачи ниже работают с `rows`.**

Сколько в выгрузке строк, участников и дней — ячейка не печатает: это вы посчитаете сами в задаче 1.

In [ ]:
# В текст запроса подставляются только известные названия таблиц и колонок —
# поэтому сначала проверяем, что значения в ячейке 1.1 из этого списка.
if TABLE not in COLUMNS:
    raise ValueError(f"TABLE должна быть одной из {list(COLUMNS)}, сейчас {TABLE!r}")
if SEGMENT_COLUMN != COLUMNS[TABLE]["segment_column"]:
    raise ValueError(f"у таблицы {TABLE} колонка сегмента называется {COLUMNS[TABLE]['segment_column']!r}")
try:
    window_days = (date.fromisoformat(PERIOD_END) - date.fromisoformat(PERIOD_START)).days + 1
except (TypeError, ValueError):
    raise ValueError("PERIOD_START и PERIOD_END — строки вида '2025-04-01'") from None

# Собираем запрос под вашу таблицу и выполняем. Обычно несколько секунд;
# если висит дольше минуты — см. «Если что-то не работает» в тексте задания.
slice_sql = SLICE_SQL.format(table=TABLE, **COLUMNS[TABLE])
rows = query(slice_sql, segment=SEGMENT, period_start=PERIOD_START, period_end=PERIOD_END)

# Защита от ошибок в значениях 1.1: пустая выгрузка или дни за пределами окна.
if not rows:
    raise ValueError("выгрузка пустая — проверьте значения в ячейке 1.1")
if len({row["day"] for row in rows}) > window_days:
    raise ValueError("в выгрузке дней больше, чем в окне — граница периода съехала")

print("Выгрузка готова. Так выглядят первые три строки:")
rows[:3]

### 1.6 Формат ответов

Служебная функция: по ней итог в части 3 проверяет, что ответ записан в нужном формате — список, число, дата строкой. **Верно ли посчитано, она не знает и не сообщает** — это проверяет преподаватель. Код ячейки свёрнут — так задумано, просто запустите её.

In [ ]:
# Какой формат ответа ожидается в каждой задаче — человеческими словами.
EXPECTED = {
    1: "list из 4 элементов: [int, int, float до 2 знаков, int]",
    2: "int",
    3: "dict: ключи из Mon…Sun, значения float до 2 знаков",
    4: "list [str, int]",
    5: "list из 3 элементов, каждый — list ['ГГГГ-ММ-ДД', float до 2 знаков]",
    6: "list [int, 'ГГГГ-ММ-ДД']",
    7: "list [float до 2 знаков, int]",
    8: "float до 2 знаков",
}


def format_problem(n: int, answer) -> str | None:
    """Что не так с ФОРМАТОМ ответа задачи n (не с правильностью). None — всё в порядке."""

    def money(x) -> bool:  # float, у которого не больше двух знаков после запятой
        return type(x) is float and round(x, 2) == x

    def iso_day(x) -> bool:  # дата строкой 'ГГГГ-ММ-ДД'
        try:
            return type(x) is str and date.fromisoformat(x).isoformat() == x
        except ValueError:
            return False

    if answer is None:
        return "не решена: функция вернула None — в ней остался ... или забыт return"
    checks = {
        1: lambda a: type(a) is list and len(a) == 4 and type(a[0]) is int and type(a[1]) is int
                     and money(a[2]) and type(a[3]) is int,
        2: lambda a: type(a) is int,
        3: lambda a: type(a) is dict and set(a) <= {"Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"}
                     and all(money(v) for v in a.values()),
        4: lambda a: type(a) is list and len(a) == 2 and type(a[0]) is str and type(a[1]) is int,
        5: lambda a: type(a) is list and len(a) == 3
                     and all(type(p) is list and len(p) == 2 and iso_day(p[0]) and money(p[1]) for p in a),
        6: lambda a: type(a) is list and len(a) == 2 and type(a[0]) is int and iso_day(a[1]),
        7: lambda a: type(a) is list and len(a) == 2 and money(a[0]) and type(a[1]) is int,
        8: lambda a: money(a),
    }
    if checks[n](answer):
        return None
    return f"нужен {EXPECTED[n]}, сейчас {answer!r}"


print("проверка формата готова")

## Часть 2. Восемь задач

У каждой задачи два шага:

1. **Посчитать.** Допишите функцию `taskN(rows)` — она должна вернуть ответ ровно в указанном формате. Имена функций и переменных `answer_N` не меняйте: по ним работа проверяется автоматически, в том числе на другом участке.
2. **Ответить на вопрос.** Для ответа почти всегда нужно досчитать одно-два числа — для этого под вопросом есть пустая ячейка. Сам ответ пишется в ячейке «✍️ Ответ на вопрос N»: дважды щёлкните по ней, **заголовок не трогайте**, замените строку курсивом своим текстом и нажмите `Shift + Enter`.

**Ячейки расчётов тоже выполняются на контрольном участке.** Считайте в них от `rows` и `answer_N`; конкретные даты, `actor_id` и сегмент руками не вписывайте — на другом участке их нет, и ячейка упадёт.

**Не получилась задача — не оставляйте падающий код.** Ячейка с ошибкой останавливает Run All, и ноутбук не проходит целиком — это минус баллы за воспроизводимость. Верните в тело функции `...`: она вернёт `None`, итог покажет «❌ не решена», а остальное проверится как обычно.

**Про ответы словами.** Код можно писать вместе с ИИ-ассистентом — это нормально, отметьте это в декларации. А ответы пишите сами: здесь оценивается, как думаете вы. **Текст, написанный LLM, получает 0 баллов.** Хороший ответ — 2–4 предложения, в них ваши числа и ваш вывод. Сомнительный вывод баллов не приносит, даже если числа верные.

**Ноутбук не скажет, правильно ли вы посчитали.** Итог в части 3 проверяет только формат. Правильность проверяет преподаватель — на вашем участке и на контрольном.

### Три особенности данных, на которых спотыкаются чаще всего

* **`value` приходит типом `Decimal`, а не `float`.** Это точный тип для денег. `Decimal` складывается с `Decimal` и с целыми числами, а с `float` — нет: `Decimal("1.5") + 0.5` падает с `TypeError`. В ноутбуке С2-идиомы суммы копились от `total = 0.0` — здесь так не выйдет. Копите суммы в `Decimal` (`defaultdict(Decimal)`, `sum(...)` без стартового `0.0`), а к `float` приводите в самом конце: `float(round(сумма, 2))`.
* **В `support_tickets` у нерешённых обращений `value` равно `None`** — времени до решения ещё нет. Проверяйте `row["value"] is not None` там, где складываете `value`, а не `if row["value"]`: второе молча выкинет и честный ноль. Сами строки с `None` из `rows` не удаляйте — это нерешённые обращения, и они нужны в задачах 1, 2 и в вопросе 1.
* **Строки с `ok = True` и отрицательным или нулевым `value` не выбрасывайте.** В `transactions` такие есть — например, покупка на ноль рублей. Считайте их как есть.

Нужные приёмы разобраны в [ноутбуке С2-идиомы](https://github.com/tikhomirovd/python-for-ba-hse-2026/blob/master/02-среда-git-python/семинар/С2-идиомы.ipynb): генераторные выражения, множества, `defaultdict` и `Counter`, `sorted` с ключом, `enumerate`, `try/except`.

### Задача 1. Паспорт участка

Руководитель начинает каждую неделю с одного вопроса: «Сколько у нас всего — и сколько из этого настоящего?» Прежде чем что-то анализировать, надо знать масштаб участка.

**Что посчитать**

Четыре числа по вашей выгрузке `rows`:

1. сколько всего строк;
2. сколько строк с `ok = True` — событий, которые состоялись;
3. сколько `value` набежало по строкам с `ok = True` — выручка, минуты или часы, смотря какая у вас таблица;
4. сколько **различных** дней встречается в выгрузке — по всем строкам, а не только по состоявшимся.

**Формат ответа:** `[всего, с_ok, сумма, дней]` — например `[2222, 2100, 123456.7, 17]`: целое, целое, `float` до двух знаков, целое.

**Подсказка:** `len(rows)`; `sum(1 for row in rows if ...)`; множество дней `{row["day"] for row in rows}`.

In [ ]:
def task1(rows: list[dict]) -> list:
    # верните [всего, с_ok, сумма, дней]
    return [
        len(rows)\
        , sum(1 for row in rows if row["ok"] == True)\
        , float(round(sum(row["value"] for row in rows if row["ok"] == True), 2))\
        , len({row["day"] for row in rows})
    ]

answer_1 = task1(rows)
answer_1

**Вопрос 1.** **Что не случилось.** Какая доля строк вашей выгрузки не состоялась (`ok = False`)? Что это значит в жизни вашей таблицы — что именно не произошло? Сколько `value` приходится на эти строки — или объясните, почему у вашей таблицы это число не имеет смысла или его нельзя посчитать.

In [ ]:
# Числа для ответа на вопрос 1 считайте здесь — по rows и answer_1.
# Ячейка тоже выполняется на контрольном участке: даты, id и сегмент руками не вписывайте.
print(1-1.0*1206/1248)
print(float(round(sum(row["value"] for row in rows if row["ok"] == False), 2)))
print(float(round(sum(row["value"] for row in rows if row["ok"] == False), 2)) / float(sum(row["value"] for row in rows)))
for row in rows:
    if row["ok"] == False:
        print(row)
print('-------------------------')
for row in rows:
    if row["actor_id"] == '1e92d85e-00ea-4d9d-af02-051bcf093cc0':
        print(row)


#### ✍️ Ответ на вопрос 1

Доля строк, которая не состоялась (ok=False) равна ~ 3,37% на общую сумму 50546,1 (3,2%). То есть, покупка клиента в партнёрском сервисе через web иногда завершается отказом, возможные причины: недостаточно средств, потеря интернет-соединения или отказ пользователя - но точно установить причину на данных нельзя. 

### Задача 2. И получилось, и нет

Самые интересные участники — те, у кого в одном окне было и «получилось», и «не получилось»: оплата сначала не прошла, а потом прошла; одну покупку отклонили, другую провели. Руководитель хочет знать, сколько таких.

**Что посчитать**

Сколько **различных** участников (`actor_id`) имеют в выгрузке хотя бы одну строку с `ok = True` **и** хотя бы одну строку с `ok = False`.

**Формат ответа:** целое число, например `123`. Ноль — тоже законный ответ.

**Подсказка:** два множества участников и операция `&` между ними.

In [ ]:
def task2(rows: list[dict]) -> int:
    # верните число участников
    true_clients = {row["actor_id"] for row in rows if row["ok"] == True}
    false_clients = {row["actor_id"] for row in rows if row["ok"] == False}
    result = len(true_clients & false_clients)
    return result

answer_2 = task2(rows)
answer_2

**Вопрос 2.** **Почему столько.** Сколько в среднем строк приходится на одного участника в вашей выгрузке? Объясните, почему ответ задачи 2 получился именно таким. Если это ноль или единица — это не ошибка: объясните, откуда он берётся. Подумайте, к чему относится сегмент: к отдельной строке (у одного участника строки могут быть с разными значениями) или к участнику целиком (у всех его строк оно одно).

In [ ]:
# Числа для ответа на вопрос 2 считайте здесь — по rows и answer_2.
# Ячейка тоже выполняется на контрольном участке: даты, id и сегмент руками не вписывайте.
count_clients = Counter(row["actor_id"] for row in rows)
avg_to_clients = sum(count_clients.values()) / len(count_clients)
print(avg_to_clients)

count_clients_true = Counter(row["actor_id"] for row in rows if row["ok"] == True)
avg_to_clients_true = sum(count_clients_true.values()) / len(count_clients_true)
print(avg_to_clients_true)

true_clients = {(row["actor_id"], row["value"]) for row in rows if row["ok"] == True}
false_clients = {(row["actor_id"], row["value"]) for row in rows if row["ok"] == False}
result = true_clients & false_clients
print(result)

#### ✍️ Ответ на вопрос 2

В среднем, на одного клиента приходится 1.248 строк, если учитывать только успешные то 1.235. Количество клиентов, у кого есть операции успешные и неуспешные - 18, что может говорить о том, что 1,8% клиентов либо сталкиваются с трудностями во время оплаты продукта, либо пытаются оплатить продукт без денег на балансе. Дополнительная проверка показала, что эти клиенты совершают успешную и неуспешную покупку на разные суммы - то есть, на выборке нет таких покупок, когда клиент после неуспешной операции, пытается снова запустить операцию с той же суммой. Либо меняется сумма во время нового заказа, либо клиент заказывает другие товары - для ответа на этот вопрос недостаточно данных.

### Задача 3. Главный день недели

Команда решает, в какой день недели ставить дежурство и запускать рассылки. Нужна раскладка участка по дням недели.

**Что посчитать**

Сумма `value` по строкам с `ok = True` — отдельно для каждого дня недели.

**Формат ответа:** словарь `{'Mon': 12345.67, 'Tue': ...}` — только дни недели, которые встретились; суммы `float` до двух знаков. Название дня берите из списка `WEEKDAYS` по номеру `row["day"].weekday()` (понедельник — 0).

**Подсказка:** `defaultdict(Decimal)`. Почему не `day.strftime('%a')`: результат зависит от языковых настроек — стоит где-нибудь вызвать `locale.setlocale`, и вместо `'Tue'` получится `'вт'`, а ответ разойдётся с форматом.

In [ ]:
WEEKDAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]  # WEEKDAYS[0] — понедельник


def task3(rows: list[dict]) -> dict:
    # верните {'Mon': сумма, 'Tue': сумма, ...}
    
    date_sum = defaultdict(Decimal)
    
    for row in rows:
        if row["ok"]:
            weekday = row["day"].weekday()
            day_name = WEEKDAYS[weekday]
    
            # здесь одна строка:
            date_sum[day_name] += row["value"]
            
    {
        day: round(float(value), 2)
        for day, value in date_sum.items()
    }

    return (
    {
        day: round(float(date_sum[day]), 2)
        for day in WEEKDAYS
        if day in date_sum
    }
    )


answer_3 = task3(rows)
answer_3

**Вопрос 3.** **Сумма или среднее.** Сколько **различных дат** вашей выгрузки приходится на каждый день недели? Считайте даты, а не строки. Какой день лидирует по сумме из задачи 3, а какой — по средней сумме на одну такую дату (сумма из задачи 3, делённая на число дат этого дня недели)? Если лидеры разные — какой ответ вы понесли бы руководителю и почему; если одинаковые — объясните, почему так вышло.

In [ ]:
# Числа для ответа на вопрос 3 считайте здесь — по rows и answer_3.
# Ячейка тоже выполняется на контрольном участке: даты, id и сегмент руками не вписывайте.
date_week_day = defaultdict(set)
date_sum = defaultdict(Decimal)
for row in rows:
    if row["ok"]:
        weekday = row["day"].weekday()
        day_name = WEEKDAYS[weekday]

        # здесь одна строка:
        date_week_day[day_name].add(row["day"])

date_week_day

for row in rows:
        if row["ok"]:
            weekday = row["day"].weekday()
            day_name = WEEKDAYS[weekday]
    
            # здесь одна строка:
            date_sum[day_name] += row["value"]

date_week_day_step1 = {
    day: round(len(days), 2)
    for day, days in date_week_day.items()
}

{
    day: round(float(date_week_day_step1[day]), 2)
    for day in WEEKDAYS
    if day in date_week_day_step1
}

{
    day: {
        "sum": round(float(date_sum[day]), 2),
        "days": len(date_week_day[day]),
        "avg": round(float(date_sum[day] / len(date_week_day[day])), 2)
    }
    for day in WEEKDAYS
    if day in date_sum
}

#### ✍️ Ответ на вопрос 3

В основном, на каждый день недели приходится по 4 дня, исключениями являются вторник и среда. По сумме лидирует среда, средний чек в этот день также выше, чем во все другие дни. Если по дням различие не сильно большое, то по среднему чеку, среда достаточно сильно отличается от других дней. Если для дежурства требуется самый активный день, то можно взять среду. 

### Задача 4. Самый активный участник

Маркетинг хочет наградить самого активного участника участка — того, у кого больше всего состоявшихся событий. Приз уже куплен.

**Что посчитать**

Участник с наибольшим числом строк с `ok = True` и это число. Если таких несколько — тот, чей `actor_id` меньше при сравнении как текст (так сравнивает `min`).

**Формат ответа:** `[actor_id, число строк]` — например `['aaaaaaaa-1111-2222-3333-bbbbbbbbbbbb', 9]`: текст (`str`) и целое.

**Подсказка:** `Counter` по строкам с `ok = True` считает, сколько состоявшихся событий у каждого участника; `max` находит наибольшее число, `min` — наименьший `actor_id` среди тех, у кого оно такое.

In [ ]:
import pandas as pd

rows_pd = pd.DataFrame(rows)

def task4(rows: list[dict]) -> list:
    # верните [actor_id, число строк]
    rows_pd = pd.DataFrame(rows)
    rows_per_user = rows_pd[rows_pd["ok"] == True].groupby("actor_id").agg(count=("value", "size")).sort_values("count", ascending=False)
    # тут айдишник пользователя стал индексом поэтому обращаться напрямую к столбцу нельзя (((((
    top_guy = rows_per_user[rows_per_user["count"] == 5].index.min()  
    return [top_guy, int(rows_per_user.loc[top_guy, "count"])]

answer_4 = task4(rows)
answer_4

**Вопрос 4.** **Кто победил на самом деле.** Сколько участников делят первое место? Если больше одного — что на самом деле выбрало вашего «победителя»: данные или правило? И что в вашей таблице вообще значит «самый активный» — стоит ли маркетингу вручать приз по такому рейтингу?

In [ ]:
# Числа для ответа на вопрос 4 считайте здесь — по rows и answer_4.
# Ячейка тоже выполняется на контрольном участке: даты, id и сегмент руками не вписывайте.
# true_clients = {row["actor_id"], row[""] for row in rows if row["ok"] == True}

rows_pd = pd.DataFrame(rows)
rows_per_user = rows_pd[rows_pd["ok"] == True].groupby("actor_id").agg(count=("value", "size")).sort_values("count", ascending=False)
rows_per_user[rows_per_user["count"] == 5]

#### ✍️ Ответ на вопрос 4

Всего 5 человек делит первое место - по 5 успешных строчек на каждого. Победителя в нашем случае выбрало правило хэширования айди пользователя, которое заложено в базе), а вообще случайным образом по набору символов. В нашей таблице, самым активным считается клиент, кто больше всего натранзачил. В целом, количество успешных транзакций говорит о самом активном участнике. Но для выбора конкретного, лучше посмотреть еще на дополнительную метрику, например сумму покупок. 

### Задача 5. Три пиковых дня

Чтобы планировать нагрузку на команду и серверы, нужно знать пиковые дни окна.

**Что посчитать**

Три даты с наибольшей суммой `value` по строкам с `ok = True` — по убыванию суммы. Если суммы равны, раньше идёт более ранняя дата.

**Формат ответа:** `[['2024-03-15', 98765.43], ['2024-03-02', 91200.1], ['2024-03-21', 90000.0]]` — дата строкой `'ГГГГ-ММ-ДД'`, сумма `float` до двух знаков.

**Подсказка:** сначала словарь «день → сумма», потом `sorted(..., key=...)`: ключ-кортеж `(-сумма, день)` сортирует по убыванию суммы, а при равенстве — по возрастанию даты. Дата строкой — `day.isoformat()`.

In [ ]:
rows_pd = pd.DataFrame(rows)

def task5(rows: list[dict]) -> list:
    # верните [['ГГГГ-ММ-ДД', сумма], [...], [...]]
    top_3_days = rows_pd[rows_pd["ok"] == True].groupby("day").agg(all_value=("value", "sum")).sort_values("all_value", ascending=False).iloc[:3]
    top_3_days = top_3_days.reset_index()
    
    return [
        [day.isoformat(), round(float(value), 2)]
        for day, value in top_3_days[["day", "all_value"]].values.tolist()
    ]

answer_5 = task5(rows)
answer_5

**Вопрос 5.** **Откуда пик.** Какие это дни недели, сколько среди них суббот и воскресений? Сформулируйте одну гипотезу, почему пик пришёлся на эти даты, и напишите, какой расчёт на данных её подтвердил бы или опроверг.

In [ ]:
# Числа для ответа на вопрос 5 считайте здесь — по rows и answer_5.
# Ячейка тоже выполняется на контрольном участке: даты, id и сегмент руками не вписывайте.
WEEKDAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
top_days = rows_pd[rows_pd["ok"] == True].groupby("day").agg(all_value=("value", "sum")).sort_values("day", ascending=True)
top_days = top_days.reset_index()
top_days["weekday"] = top_days["day"].apply(
    lambda day: WEEKDAYS[day.weekday()]
)
top_days.groupby("weekday")\
    .agg(all_value_per_day=("all_value", "sum"), avg_value=("all_value", "mean"), all_transactions=("day", "count"))\
    .sort_values("all_value_per_day", ascending=False)

#### ✍️ Ответ на вопрос 5

Пик приходится на среду - больше всего сумма, а также средняя сумма по средам выше, чем в другие дни. Возможно партнеры, в которых производится оплата, запускают какие-то коммуникации по средам. Но в целом выводы сделать сложно на основании одного месяца, так как транзакций во вторник столько же, сколько и в среду - различаются только суммы. В субботу и воскресенье по 4 транзакции. Гипотеза про коммуникации проверяется с помощью таблицы коммуникаций или промокампаний партнеров. 

### Задача 6. Когда набралась половина

Финансам важно, равномерно ли набирается сумма внутри окна: к какому дню набегает половина суммы по состоявшимся событиям.

**Что посчитать**

1. Возьмите дни, в которых есть хотя бы одна строка с `ok = True`, и расставьте по возрастанию даты.
2. Идите по ним и копите сумму `value` по строкам с `ok = True`.
3. Найдите первый день, на котором накопленное **достигло половины** суммы из пункта 3 задачи 1 (всё `value` по строкам с `ok = True`) или превысило её, и его порядковый номер в списке дней из шага 1, считая с 1.

**Формат ответа:** `[номер, 'ГГГГ-ММ-ДД']` — например `[12, '2024-03-15']`.

**Подсказка:** `sums` — словарь «день → сумма», как в задаче 5. `for number, day in enumerate(sorted(sums), start=1):` — `enumerate` сам считает номер.

In [ ]:
def task6(rows: list[dict]) -> list:
    # верните [номер дня, 'ГГГГ-ММ-ДД']
    top_days = rows_pd[rows_pd["ok"] == True].groupby("day").agg(all_value=("value", "sum")).sort_values("day", ascending=True)
    top_days["nakop_sum"] = (
        top_days["all_value"].cumsum()
    )
    top_days["nakop_day"] = range(1, len(top_days) + 1)
    
    top_days["first_day_more_half"] = (top_days["nakop_sum"] >= sum(row["value"] for row in rows if row["ok"] == True)/2).astype(int)
    
    result = top_days[top_days["first_day_more_half"] == 1].sort_values("day", ascending=True).iloc[0]
    
    return [
        int(result["nakop_day"]),
        result.name.isoformat()
    ]


answer_6 = task6(rows)
answer_6

**Вопрос 6.** **Равномерно ли.** Какую долю дней заняла первая половина суммы: номер дня из задачи 6, делённый на число дней в списке из шага 1? Номер дня целый, поэтому при равномерном накоплении получилось бы около половины, округлённой вверх (15 из 30, 16 из 31) — сравнивайте с этим, а не с ровными 50 %. У вас раньше или позже — и что это говорит о том, как менялся участок внутри окна? Если вышло у середины — так и напишите.

In [ ]:
# Числа для ответа на вопрос 6 считайте здесь — по rows и answer_6.
# Ячейка тоже выполняется на контрольном участке: даты, id и сегмент руками не вписывайте.

top_days = rows_pd[rows_pd["ok"] == True].groupby("day").agg(all_value=("value", "sum")).sort_values("day", ascending=True)
top_days["nakop_sum"] = (
    top_days["all_value"].cumsum()
)
top_days["nakop_day"] = range(1, len(top_days) + 1)

top_days["first_day_more_half"] = (top_days["nakop_sum"] >= 1524673.22/2).astype(int)

result = top_days[top_days["first_day_more_half"] == 1].sort_values("day", ascending=True).iloc[0]

answer = [
    result.name.isoformat(),
    int(result["nakop_day"])
]

top_days

#### ✍️ Ответ на вопрос 6

Половина суммы набралась после 15го дня месяца. Так как в апреле всего 30 дней, на моей выборке транзакций, попадание ровно в середину. Итого получается 15/30 или 50% дней. Это может говорить о том, что в целом нет какой-либо тенденции на оплату в начале месяца или в конце - транзакции распределены без перекоса.

### Задача 7. Функция, которая не падает

Соседний отдел присылает суммы строками — так, как их набрал оператор: с запятой, с пробелами, иногда с прочерком. Нужна функция, которая превращает такую строку в число и не роняет ежедневный отчёт. Данные в этой задаче у всех одинаковые — список `RAW`.

**Что посчитать**

1. Функция `to_value(x)`: получает строку (или `None`) и возвращает `float` — или `None`, если число разобрать нельзя. Падать нельзя ни на одном значении. Правила записи:
   * точка и запятая — десятичный разделитель: `'89.10'` → `89.1`, `'349,90'` → `349.9`;
   * пробел и неразрывный пробел внутри числа — разделители тысяч: `'1 499,50'` → `1499.5`;
   * пробелы по краям ничего не значат;
   * прочерк `'—'` и пустая строка — не число: `None`, а не `0.0`.
2. По списку `RAW`: сумма всего, что разобралось, и сколько значений отброшено. `None` тоже считается отброшенным.

**Формат ответа:** `[сумма, отброшено]` — `float` до двух знаков и целое.

**Подсказка:** `try: ... except ValueError: ...` — ловите именно `ValueError`, а не всё подряд. `None` проверьте до `try`: `float(None)` бросает `TypeError`, а не `ValueError`. `x.split()` делит строку по любым пробелам, включая неразрывный.

In [ ]:
RAW = ["199.00", "1 499,50", None, "—", "349,90", "2\u00a0100", "", "  89.10  "]

# Во втором значении обычный пробел, в шестом — неразрывный (\u00a0).
# Если assert упал — RAW испортился при копировании: верните ячейку из стартового ноутбука.
assert RAW[1][1] == " " and RAW[5][1] == "\u00a0"


def to_value(x: str | None) -> float | None:
    # верните число или None; падать нельзя
    if x is None:
        return None

    x = str(x).strip() # срезает пробелы спраа и слева

    if x in ("", "—", "-"):
        return None

    # тут сначала сплит значения на пробелы, соединяет их, и меняет запятую на точку
    x = "".join(x.split()).replace(",", ".")

    try:
        return float(x)
    except ValueError:
        return None


def task7(raw: list) -> list:
    # верните [сумма разобранного, сколько отброшено]
    values = pd.Series(raw, dtype="object").map(to_value)

    total = round(float(values.sum()), 2)
    dropped = int(values.isna().sum())

    return [total, dropped]


answer_7 = task7(RAW)
answer_7

**Вопрос 7.** **Что опаснее.** Представьте ежедневный отчёт о выручке, построенный на этой функции. Что опаснее: функция, которая падает на `'—'`, или функция, которая молча возвращает `None`? Что бы вы добавили в отчёт, чтобы потеря была видна руководителю? Опирайтесь на свой ответ задачи 7: сколько из восьми значений было бы потеряно.

In [ ]:
# Числа для ответа на вопрос 7 считайте здесь — по rows и answer_7.
# Ячейка тоже выполняется на контрольном участке: даты, id и сегмент руками не вписывайте.
RAW = ["199.00", "1 499,50", None, "—", "349,90", "2\u00a0100", "", "  89.10  "]



#### ✍️ Ответ на вопрос 7

Лучше функция, которая падает на '-' чем функция, которая молча игнорирует такие значения. Иначе можно не понять, что есть какая-то проблема, и например 80% транзакций сейчас не учитывается в отчете. Лучше чтобы была какая-то метрика, которая показывает, сколько значений не было учтено в итоговом отчете. Например у меня получилось 3/8 = 37,5%

### Задача 8. Поправка на день недели

Аналитики из соседней команды прислали коэффициенты дней недели, чтобы сравнивать недели между собой. Прислали только будни — как это обычно и бывает.

**Что посчитать**

1. Для каждого дня недели возьмите сумму `value` по строкам с `ok = True` — как в задаче 3, но **до округления**.
2. Приведите её к `float` и умножьте на коэффициент дня из `WEIGHTS`. Если дня в справочнике нет — коэффициент `1.0`.
3. Сложите все произведения. Округлите до двух знаков **только итог**.

**Формат ответа:** одно число `float` до двух знаков, например `123456.78`.

**Подсказка:** `WEIGHTS.get(day, 1.0)` возвращает коэффициент или `1.0`, если дня нет. `Decimal * float` падает с `TypeError` — сначала `float(...)`.

In [ ]:
WEIGHTS = {"Mon": 1.00, "Tue": 1.05, "Wed": 1.05, "Thu": 1.00, "Fri": 0.95}


def task8(rows: list[dict]) -> float:
    # верните взвешенную сумму
    WEEKDAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

    week_days = rows_pd.copy()
    
    week_days["week_day"] = pd.to_datetime(week_days["day"]).dt.dayofweek.apply(
        lambda x: WEEKDAYS[x]
    )
    
    week_days = week_days[week_days["ok"] == True].groupby("week_day").agg(all_value=("value", "sum")).sort_values("week_day", ascending=True)
    week_days = week_days.reset_index()
    week_days["weight"] = week_days["week_day"].map(WEIGHTS).fillna(1.0)
    total_value = (week_days["all_value"].astype(float)*week_days["weight"]).sum()
    return round(float(total_value), 2)


answer_8 = task8(rows)
answer_8

**Вопрос 8.** **Что изменила поправка.** Какие дни недели пошли с коэффициентом по умолчанию? На сколько процентов (с точностью до десятых) взвешенная сумма отличается от обычной суммы из задачи 1 — процент считайте от обычной суммы? В какой ситуации на данных ПРАЙМ такой `.get` с умолчанием спрятал бы ошибку, вместо того чтобы о ней сообщить?

In [ ]:
# Числа для ответа на вопрос 8 считайте здесь — по rows и answer_8.
# Ячейка тоже выполняется на контрольном участке: даты, id и сегмент руками не вписывайте.
WEEKDAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

week_days = rows_pd.copy()

week_days["week_day"] = pd.to_datetime(week_days["day"]).dt.dayofweek.apply(
    lambda x: WEEKDAYS[x]
)


week_days = week_days[week_days["ok"] == True].groupby("week_day").agg(all_value=("value", "sum")).sort_values("week_day", ascending=True)
week_days = week_days.reset_index()
week_days["weight"] = week_days["week_day"].map(WEIGHTS).fillna(1.0)
total_value = (week_days["all_value"].astype(float)*week_days["weight"]).sum()
total_value
total_value/1524673.22

#### ✍️ Ответ на вопрос 8

Со стандартным коэфициентом были: понедельник, четверг, суббота и воскресенье. Новая сумма отличается на ~1,2%. Если бы день отсутсвовал в данных, то тогда бы для него все равно был коэффициент 1.0. То есть, если бы к нам попала строка, например ('', true, 300), то мы бы все равно посчитали ее в тотал сумме, даже если бы не определился день недели (но группировка по дням недели бы показала, None - 300)

## Часть 3. Итог — готовый код

При **Restart Kernel and Run All Cells** эта ячейка выполняется после всех задач: перед сдачей посмотрите её вывод. Она показывает ваши ответы и проверяет только формат, но не правильность.

Последняя строка вывода — служебная: по ней преподаватель сверяет ответы автоматически. Ячейку не удаляйте.

In [ ]:
import json


def typed(x):
    """Для служебной строки: всё, кроме list/dict/str/int/float/bool/None, помечаем типом."""
    if type(x) is dict:
        return {str(key): typed(value) for key, value in x.items()}
    if type(x) is list:
        return [typed(value) for value in x]
    if x is None or type(x) in (bool, int, float, str):
        return x
    return f"<{type(x).__name__}> {x!r}"


# Участок из ячейки 1.1 и ответы answer_1 … answer_8; у каждого ответа проверяем формат.
params = {"table_name": TABLE, "segment_column": SEGMENT_COLUMN, "segment": SEGMENT,
          "period_start": PERIOD_START, "period_end": PERIOD_END}
answers = {n: globals().get(f"answer_{n}") for n in range(1, 9)}
formats = {n: format_problem(n, answer) for n, answer in answers.items()}

for n, answer in answers.items():
    print(f"задача {n}: {answer!r}")
    print("   ✅ формат в порядке" if formats[n] is None else f"   ❌ {formats[n]}")

print("\nНе забудьте ответы словами — ячейки «✍️ Ответ на вопрос N» в части 2.")
print("\n--- служебная строка, не удаляйте ---")
print("HW1_ANSWERS=" + json.dumps(
    {"slice": params, "answers": typed(answers), "format": formats}, ensure_ascii=False
))

## Часть 4. Исследование — для тех, кто хочет 9 или 10

**Эта часть необязательна.** Части 1–3 — это оценка до 8 баллов, и 8 — это «отлично». 9 и 10 ставятся только за исследование: два задания, каждое до 1 балла сверху. Засчитываются, если основная часть набрала не меньше 15 баллов из 20.

Здесь нет готового формата ответа и заготовок. Вы сами решаете, что посчитать, считаете это кодом в ячейках ниже и пишете вывод. Ассистент может помочь с кодом, но ответ зависит от того, что вы увидите в своих данных. Текст вывода — ваш: правило про LLM действует и здесь.

| Каждое исследование оценивается так | Балл |
|---|---|
| числа верные и получены кодом в ноутбуке, а не вписаны руками | 0,4 |
| вывод следует из чисел, названо, чего ваш расчёт **не** доказывает | 0,3 |
| код исследования выполняется у преподавателя целиком — в том числе на контрольном участке | 0,2 |
| изложено по схеме: вопрос → как проверяли → что получилось → что это значит | 0,1 |

Код исследования, как и всё остальное, считайте от значений ячейки 1.1: на контрольном участке он должен отработать без правок.

База умеет считать сама: `count(*)`, `count(distinct …)`, `sum(…)`, условие внутри агрегата — `count(*) filter (where …)`, разбивка — `group by`. Весь сегмент в Python не выгружайте: запрос дольше 2 минут сервер оборвёт. Функция `query` и словарь `COLUMNS` из части 1 здесь пригодятся.

### Исследование А. Можно ли верить вашей тысяче?

Руководитель прочитал сводку и спрашивает: «Вы смотрели тысячу участников, а в сегменте их гораздо больше. Насколько ваши числа верны для всего сегмента? Если в следующем месяце value на участника упадёт на 5 %, мы это заметим по такой тысяче?»

**Что сделать**

1. По **всему** вашему сегменту за окно — одним запросом к базе, не выгружая строки — посчитайте: сколько участников, долю строк с `ok = True`, `value` на участника (сумма `value` по строкам с `ok = True`, делённая на число различных участников).
2. Те же показатели — по вашей тысяче, из `rows`.
3. Возьмите **не меньше 20 других тысяч** участников того же сегмента и окна и посчитайте показатели для каждой.
4. Ответьте: насколько ваша тысяча отличается от всего сегмента? Каков разброс между тысячами? Заметно ли по одной тысяче падение `value` на участника на 5 % — и сколько участников нужно брать, чтобы такое падение было заметно?

**Подсказки.** Другая тысяча — это другое перемешивание. Ячейку 1.4 не меняйте — сделайте копию запроса в своей ячейке: `salted_sql = SLICE_SQL.replace("md5(actor_id)", "md5(actor_id || :salt)").format(table=TABLE, **COLUMNS[TABLE])` и вызывайте `query(salted_sql, segment=SEGMENT, period_start=PERIOD_START, period_end=PERIOD_END, salt=str(i))` с разными `i`. Разброс удобно описать стандартным отклонением (`statistics.pstdev`). Как разброс зависит от размера выборки — поищите «стандартная ошибка среднего».

In [ ]:
# Исследование А: ваш код. Всё — от значений ячейки 1.1.
# Собираем семпл из 20 тыс 
salted_sql = SLICE_SQL.replace(
    "md5(actor_id)",
    "md5(actor_id || :salt)"
).format(
    table=TABLE,
    **COLUMNS[TABLE]
)

samples = []

for i in range(1, 21):
    sample = query(
        salted_sql,
        segment=SEGMENT,
        period_start=PERIOD_START,
        period_end=PERIOD_END,
        salt=str(i)
    )

    samples.append(sample)

In [ ]:
# Считаем метрики для первой дополнительной тысячи 
import pandas as pd

df = pd.DataFrame(samples[0])

n_clients = df["actor_id"].nunique()

ok_share = df["ok"].mean()

# Сумма успешных операций
total_value = df.loc[df["ok"], "value"].sum()

# Value на одного участника
value_per_client = float(total_value) / n_clients

print("Участников:", n_clients)
print("Доля успешных строк:", ok_share)
print("Сумма на участника:", value_per_client)

In [ ]:
# Функция для расчета параметров для каждой тысячи
def calc_metrics(sample):
    df = pd.DataFrame(sample)

    n_clients = df["actor_id"].nunique()
    ok_share = df["ok"].mean()

    total_value = df.loc[df["ok"], "value"].sum()
    value_per_client = float(total_value) / n_clients

    return [n_clients, ok_share, value_per_client]

results = []

for i, sample in enumerate(samples, start=1):
    metrics = calc_metrics(sample)

    results.append([i] + metrics)


results_pd = pd.DataFrame(
    results,
    columns=[
        "sample",
        "n_clients",
        "ok_share",
        "value_per_client"
    ]
)

# дескрайб для среднего отклонения
results_pd[
    ["ok_share", "value_per_client"]
].describe()

In [ ]:
# считаем мою тыщу

original_metrics = calc_metrics(rows)

original_metrics

In [ ]:
original_value = original_metrics[2]

mean_value = results_pd["value_per_client"].mean()

difference_pct = (
    original_value / mean_value - 1
) * 100

print("Исходная тысяча:", original_value)
print("Среднее по 20 выборкам:", mean_value)
print("Отклонение, %:", difference_pct)

In [ ]:
# запрос по всей базе
SEGMENT_SQL = """
with base AS 
(
    select
        {actor}::text AS actor_id
        , {ok} AS ok
        , {value} AS value
    from prime.{table}
    where 1=1
    and {segment_column} = :segment
    and {moment} >= cast(:period_start AS date)
    and {moment} < cast(:period_end AS date) + 1
)
select
    count(distinct actor_id) AS n_clients
    , avg(
        case when ok then 1.0 else 0.0 end
    ) AS ok_share
    , sum(
        case when ok then value else 0 end
    ) / nullif(count(distinct actor_id), 0) as value_per_client
from base
"""

In [ ]:
segment_sql = SEGMENT_SQL.format(
    table=TABLE,
    **COLUMNS[TABLE]
)

segment_result = query(
    segment_sql,
    segment=SEGMENT,
    period_start=PERIOD_START,
    period_end=PERIOD_END
)

# Метрики по всему сегменту
segment_metrics = segment_result[0]

segment_clients = segment_metrics["n_clients"]
segment_ok_share = float(segment_metrics["ok_share"])
segment_value = float(segment_metrics["value_per_client"])

print(segment_clients)
print(segment_ok_share)
print(segment_value)

In [ ]:
# Отклонение исходной тысячи от всего сегмента
original_diff = (
    original_value / segment_value - 1
) * 100

# Отклонение среднего по 20 выборкам
samples_diff = (
    mean_value / segment_value - 1
) * 100

print("Исходная тысяча:", original_diff)
print("Среднее по 20 выборкам:", samples_diff)

In [ ]:
std_1000 = results_pd["value_per_client"].std()

std_4000 = std_1000 * (1000 / 4000) ** 0.5

print(std_4000)

In [ ]:
# сколько выборок также отклоняется больше чем на 5%
results_pd["deviation_pct"] = (
    results_pd["value_per_client"] / segment_value - 1
) * 100

n_large_deviations = (
    results_pd["deviation_pct"].abs() >= 5
).sum()

print(n_large_deviations)

In [ ]:
# какая примерно выборка нужна, чтобы хватило отследить 5% эффект
std_1000 = results_pd["value_per_client"].std()

drop_value = segment_value * 0.05

n_required = 1000 * (2 * std_1000 / drop_value) ** 2

print("Размер выборки:", round(n_required))

#### ✍️ Исследование А
--Насколько ваша тысяча отличается от всего сегмента? 

Для проверки были сформированы 20 дополнительных выборок по 1000 строк. Если взять 20000 строк других выборок, то в них значение value в среднем на участника составляет 1664.3.
В исходной базе этот показатель 1524,67. Отклонение  составляет -8.39%. Получается, что результаты одной тысячи могут сильно отклоняться от результата по большей выборке.

--Каков разброс между тысячами? Заметно ли по одной тысяче падение `value` на участника на 5 % — и сколько участников нужно брать, чтобы такое падение было заметно?

Стандартное отклонение при этом составило 70,15.
В 5 из 20 выборок (25%) отклонение от всего сегмента составило не менее 5% по среднему value. Это говорит нам о том, что стандартное отклонение выборки на 1000 может быть сопоставимо с изменением показателя, которое мы хотим обнаружить.

Если снизить средний value на 5% от показателя всего сегмента то получится 82,66. Стандартное отклонение показателя между выборками по 1000 участников получилось 70,15.
Получается, что предполагаемое снижение практически такое же как и разброс между выборками. 

Для достижения точности, при которой две стандартные ошибки оценки одного месяца соответствуют изменению на 5%, необходимо около 2900 участников.

### Исследование Б. Ваш сегмент особенный?

Маркетинг заметил, что в одних сегментах участники приносят заметно больше, чем в других, и хочет перераспределить бюджет в пользу «ценных» сегментов. Руководитель просит проверить: правда ли ваш сегмент отличается от соседних — и чем именно.

**Что сделать**

1. Для **всех сегментов** вашей таблицы за ваше окно посчитайте: число участников, строк на участника, долю строк с `ok = True`, `value` на одну строку с `ok = True` и `value` на участника.
2. Разложите `value` на участника на множители и найдите, какой из них даёт различие между сегментами.
3. Проверьте, держится ли картина в предыдущем окне той же длины. Данные начинаются 6 января 2025 года: если предыдущее окно начинается раньше, оно неполное — тогда сравнивайте доли и значения на участника или на строку, а не число участников и строк.
4. Вывод для маркетинга: «ценный» ли ваш сегмент; что на самом деле различается; какое решение по бюджету из этого следует — и чего ваши данные не доказывают.

**Подсказка.** Здесь нужны не выгрузка `rows`, а агрегаты по всей таблице за окно с разбивкой `group by` колонке сегмента.

In [ ]:
# Исследование Б: ваш код. Всё — от значений ячейки 1.1.
# Выгружаем агрегаты по всем сегментам
SEGMENTS_SQL = """
    select
        {segment_column} as segment
        , count(distinct {actor}) as n_clients
        , count(*) as n_rows
        , count(*) filter (
            WHERE {ok}
        ) AS n_ok
        , sum({value}) filter (
            where {ok}
        ) as total_value
    from prime.{table}
    where 1=1
    and {moment} >= cast(:period_start AS date)
    and {moment} < cast(:period_end AS date) + 1
    group by {segment_column}
    order by {segment_column}
"""

segments_sql = SEGMENTS_SQL.format(
    table=TABLE,
    **COLUMNS[TABLE]
)

segments_data = query(
    segments_sql,
    period_start=PERIOD_START,
    period_end=PERIOD_END
)

In [ ]:
segments_data

In [ ]:
# Считаем основные показатели для каждого сегмента
# количество клиентов, долю успешных операций и среднюю сумму на операцию и на клиента

segments_pd = pd.DataFrame(segments_data)

# Количество строк на одного клиента
segments_pd["rows_per_client"] = (
    segments_pd["n_rows"] / segments_pd["n_clients"]
)

# Доля успешных строк
segments_pd["ok_share"] = (
    segments_pd["n_ok"] / segments_pd["n_rows"]
)

# Сумма на одну успешную строку
segments_pd["value_per_ok"] = (
    segments_pd["total_value"] / segments_pd["n_ok"]
)

# Сумма на одного клиента
segments_pd["value_per_client"] = (
    segments_pd["total_value"] / segments_pd["n_clients"]
)

print(segments_pd)

In [ ]:
# Проверяем, что произведение трёх множителей совпадает с исходным показателем value на клиента
segments_pd["value_check"] = (
    segments_pd["rows_per_client"]
    * segments_pd["ok_share"]
    * segments_pd["value_per_ok"].astype(float)
)

print(segments_pd)

In [ ]:
# Сравниваем показатели других сегментов с нашим web

compare = segments_pd.set_index("segment")

metrics = [
    "rows_per_client",
    "ok_share",
    "value_per_ok",
    "value_per_client"
]

compare_pct = (
    compare[metrics].astype(float)
    .div(compare.loc["web", metrics].astype(float))
    .sub(1)
    .mul(100)
    .round(2)
)

compare_pct

In [ ]:
# Определяем предыдущий период такой же продолжительности
# Выгружаем по всем сегментам для того чтобы понять сохраняются ли различия между ними

from datetime import datetime, timedelta

period_start_dt = datetime.strptime(PERIOD_START, "%Y-%m-%d").date()
period_end_dt = datetime.strptime(PERIOD_END, "%Y-%m-%d").date()

# Продолжительность текущего периода
period_days = (period_end_dt - period_start_dt).days + 1

# Предыдущий период такой же продолжительности
PREV_PERIOD_END = period_start_dt - timedelta(days=1)
PREV_PERIOD_START = PREV_PERIOD_END - timedelta(days=period_days - 1)

print("Текущий период:", period_start_dt, PERIOD_END)
print("Предыдущий период:", PREV_PERIOD_START, PREV_PERIOD_END)

prev_segments_data = query(
    segments_sql,
    period_start=PREV_PERIOD_START,
    period_end=PREV_PERIOD_END
)

prev_segments_pd = pd.DataFrame(prev_segments_data)

prev_segments_pd

In [ ]:
# Рассчитываем показатели сегментов за предыдущий период

prev_segments_pd["rows_per_client"] = (
    prev_segments_pd["n_rows"] / prev_segments_pd["n_clients"]
)

prev_segments_pd["ok_share"] = (
    prev_segments_pd["n_ok"] / prev_segments_pd["n_rows"]
)

prev_segments_pd["value_per_ok"] = (
    prev_segments_pd["total_value"] / prev_segments_pd["n_ok"]
)

prev_segments_pd["value_per_client"] = (
    prev_segments_pd["total_value"] / prev_segments_pd["n_clients"]
)

prev_segments_pd

In [ ]:
# Объединяем показатели двух периодов по сегменту

compare_periods = segments_pd[
    ["segment", "rows_per_client", "ok_share",
     "value_per_ok", "value_per_client"]
].merge(
    prev_segments_pd[
        ["segment", "rows_per_client", "ok_share",
         "value_per_ok", "value_per_client"]
    ],
    on="segment",
    suffixes=("_apr", "_mar")
)

compare_periods

In [ ]:
# Рассчитываем относительное изменение показателя value на клиента между мартом и апрелем

compare_periods["value_change_pct"] = (
    compare_periods["value_per_client_apr"].astype(float)
    / compare_periods["value_per_client_mar"].astype(float)
    - 1
) * 100

compare_periods[
    ["segment",
     "value_per_client_mar",
     "value_per_client_apr",
     "value_change_pct"]
]

In [ ]:
# Сравниваем показатели марта относительно сегмента web

compare_mar = prev_segments_pd.set_index("segment")

metrics = [
    "rows_per_client",
    "ok_share",
    "value_per_ok",
    "value_per_client"
]

compare_mar_pct = (
    compare_mar[metrics].astype(float)
    .div(compare_mar.loc["web", metrics].astype(float))
    .sub(1)
    .mul(100)
    .round(2)
)

compare_mar_pct

In [ ]:
compare_periods[
    [
        "segment",
        "rows_per_client_mar",
        "rows_per_client_apr",
        "ok_share_mar",
        "ok_share_apr",
        "value_per_client_mar",
        "value_per_client_apr",
        "value_change_pct"
    ]
].round(3)

In [ ]:
# Рассчитываем изменение каждого множителя между периодами

factors = [
    "rows_per_client",
    "ok_share",
    "value_per_ok"
]

for factor in factors:
    compare_periods[f"{factor}_change_pct"] = (
        compare_periods[f"{factor}_apr"].astype(float)
        / compare_periods[f"{factor}_mar"].astype(float)
        - 1
    ) * 100

compare_periods[
    [
        "segment",
        "rows_per_client_change_pct",
        "ok_share_change_pct",
        "value_per_ok_change_pct",
        "value_change_pct"
    ]
].round(2)

#### ✍️ Исследование Б

Действительно, сегменты между собой различаются. 
Наш сегмент web в апреле имеет следующие параметры: 1653 сумма в среднем на клиента и 96,5% успешных операций. 
В сегменте app 2018 и 96,8%, а в сегменте pos 1541 и 96,8% успешных операций. Если доля успешных операций примерно соотносится, то средний чек на клиента различается достаточно сильно - в app он больше на 22%, а в pos он меньше на ~7%. 
Сильнее всего, на показатель в app влияет количество строк (транзакций) на клиента 1,55 против 1,27 в web, хотя средний чек на одну транзакцию практически не отличается. Это может говорить о том, что клиенты в app более склонны на совершение второй покупки, чем в web. Возможно это влияние коммуникаций в приложении - push или смс, но точно сказать нельзя.
Дополнительно посмотрел на более раннем отрезке - выводы сохраняются (различие в средних не более 3% по сравнению с предыдущим периодом).
Отвечая на вопрос маркетинга, необходимо понимать, какая стоимость привлечения клиента в каждый канал - насколько канал app дешевле или дороже канала web в обслуживании и т.д. Оценка сегмента по сумме транзакций и среднему недостаточна для принятия решения о бюджетах. Здесь может также влиять и соцдем клиентов в разных каналах - например, то что заказывают клиенты в основном моложе тех, кто ходит в магазин очно. 

## Напоследок

Что вас удивило в вашем участке? Одна-две фразы в ячейке ниже — по желанию и без баллов.

И пройдите, пожалуйста, **[короткий опрос о том, как вам это задание](https://forms.gle/5tNT4P4diHVyzhdFA)** — пара минут. Следующие задания соберу с учётом ответов.

#### ✍️ Что удивило

_По желанию._